# Faruq-v3 — AF2 × D-FINE-N cross-architecture seed-42 screen

Paired validation-only experiment:

`official D-FINE-N COCO -> DFN0` versus `official D-FINE-N COCO -> frozen AF2 -> DFN_AF2`.

The formal proposal is **not** changed by this notebook. The Faruq-v3 locked test remains absent/closed.

Frozen upstream D-FINE commit: `956d1709314c2c6a4df6f34de232054578a7449f`.

Single-GPU profile: batch 16 for both arms, linearly scaled D-FINE optimizer/warm-up/EMA settings, 220 maximum epochs, AMP, seed 42.

Required Kaggle input: exactly one `faruq-development-v3-grouped.tar.bin`.

Recommended runtime: GPU + Internet ON.


In [ ]:
from pathlib import Path
import hashlib, importlib, json, os, shutil, subprocess, sys, time, urllib.request, zipfile

INPUT=Path('/kaggle/input')
WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir():
    raise RuntimeError('Kaggle-only notebook')
os.chdir(WORK)

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

matches=sorted(INPUT.rglob('faruq-development-v3-grouped.tar.bin'))
if len(matches)!=1:
    raise FileNotFoundError(f'Harus tepat satu faruq-development-v3-grouped.tar.bin; ditemukan {matches}')
ARCHIVE=matches[0]
print('DATA INPUT:', ARCHIVE)


In [ ]:
BRANCH='agent/af2-dfine-n-transfer-screen'
COFFEE=WORK/'coffee-bean-detection'
DFINE=WORK/'D-FINE'
DFINE_COMMIT='956d1709314c2c6a4df6f34de232054578a7449f'

for path in (COFFEE, DFINE):
    if path.exists():
        shutil.rmtree(path)

subprocess.run([
    'git','clone','--depth','1','--branch',BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git',str(COFFEE)
],check=True)

subprocess.run([
    'git','clone','--depth','1',
    'https://github.com/Peterande/D-FINE.git',str(DFINE)
],check=True)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=DFINE,text=True).strip()
if head!=DFINE_COMMIT:
    subprocess.run(['git','fetch','origin',DFINE_COMMIT,'--depth','1'],cwd=DFINE,check=True)
    subprocess.run(['git','checkout','--detach',DFINE_COMMIT],cwd=DFINE,check=True)
head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=DFINE,text=True).strip()
if head!=DFINE_COMMIT:
    raise RuntimeError(f'D-FINE commit mismatch: {head}')

subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(COFFEE)],check=True)
subprocess.run([
    sys.executable,'-m','pip','install','-q',
    'faster-coco-eval>=1.6.6','tensorboard','scipy','calflops','transformers','loguru','PyYAML'
],check=True)

if str(COFFEE/'src') not in sys.path:
    sys.path.insert(0,str(COFFEE/'src'))
importlib.invalidate_caches()

print('COFFEE BRANCH:',BRANCH)
print('COFFEE COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=COFFEE,text=True).strip())
print('D-FINE COMMIT:',head)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Aktifkan Kaggle GPU')
print('GPU:',torch.cuda.get_device_name(0))
print('TORCH:',torch.__version__)


In [ ]:
from coffee_detector.experiments.prepare_faruq_v3_kaggle import prepare_faruq_v3_kaggle_input

DATA,CORE=prepare_faruq_v3_kaggle_input(INPUT,WORK)
if CORE.get('decision')!='PASS' or CORE.get('test_images_accessed') is not False:
    raise RuntimeError('Dataset contract gagal')
if (DATA/'test').exists():
    raise RuntimeError('TEST TEREXPOSE — STOP')

DATA_YAML=DATA/'data.yaml'
print('DATA:',DATA)
print('TRAIN:',CORE['splits']['train'])
print('VAL:',CORE['splits']['val'])
print('TEST ABSENT:',not (DATA/'test').exists())


In [ ]:
OUT=WORK/'af2-dfine-n-transfer-seed42-v1'
STATE_ZIP=WORK/'af2-dfine-n-transfer-seed42-state.zip'

# Restore prior paired-training state if supplied as a Kaggle input.
resume_states=sorted(INPUT.rglob(STATE_ZIP.name))
if len(resume_states)>1:
    raise RuntimeError(f'Resume state ambigu: {resume_states}')
if len(resume_states)==1 and not OUT.exists():
    print('RESTORE STATE:',resume_states[0])
    with zipfile.ZipFile(resume_states[0],'r') as z:
        z.extractall(WORK)
OUT.mkdir(parents=True,exist_ok=True)

COCO_DIR=WORK/'faruq-v3-dfine-coco'
PREP=OUT/'dfine_transfer_preparation.json'

from scripts.prepare_dfine_af2_transfer import prepare
prep=prepare(
    dfine_root=DFINE,
    data_yaml=DATA_YAML,
    coco_output=COCO_DIR,
    run_output=OUT,
    report_path=PREP,
)

NATIVE_CFG=Path(prep['configs']['DFN0'])
AF2_CFG=Path(prep['configs']['DFN_AF2'])

# Apply the prospectively frozen 16-GB-class single-GPU profile to both arms.
subprocess.run([
    sys.executable,str(COFFEE/'scripts/apply_dfine_single_gpu_profile.py'),
    str(NATIVE_CFG),str(AF2_CFG)
],check=True,cwd=COFFEE)

print('NATIVE CONFIG:',NATIVE_CFG)
print('AF2 CONFIG:',AF2_CFG)
print('COCO TRAIN SHA:',prep['dataset']['splits']['train']['sha256'])
print('COCO VAL SHA:',prep['dataset']['splits']['val']['sha256'])


In [ ]:
PRETRAIN=WORK/'dfine_n_coco.pth'
PRETRAIN_URL='https://github.com/Peterande/storage/releases/download/dfinev1.0/dfine_n_coco.pth'
if not PRETRAIN.is_file():
    print('DOWNLOAD OFFICIAL D-FINE-N COCO CHECKPOINT...')
    urllib.request.urlretrieve(PRETRAIN_URL, PRETRAIN)
if PRETRAIN.stat().st_size != 15489558:
    raise RuntimeError(f'Unexpected D-FINE-N checkpoint bytes: {PRETRAIN.stat().st_size}')
PRETRAIN_SHA=sha256(PRETRAIN)
print('D-FINE-N CHECKPOINT:',PRETRAIN)
print('BYTES:',PRETRAIN.stat().st_size)
print('SHA256:',PRETRAIN_SHA)


In [ ]:
STATIC=OUT/'dfine_af2_static_preflight.json'
cmd=[
    sys.executable,str(COFFEE/'scripts/preflight_dfine_af2_transfer.py'),
    '--dfine-root',str(DFINE),
    '--native-config',str(NATIVE_CFG),
    '--af2-config',str(AF2_CFG),
    '--pretrained-checkpoint',str(PRETRAIN),
    '--preparation-report',str(PREP),
    '--output',str(STATIC),
    '--seed','42',
]
r=subprocess.run(cmd,cwd=COFFEE,text=True,capture_output=True)
print(r.stdout)
if r.returncode:
    print(r.stderr)
    raise RuntimeError(f'STATIC PREFLIGHT FAILED rc={r.returncode}')
static=json.loads(STATIC.read_text())
failed=[k for k,v in static['gates'].items() if not v]
if failed or static['decision']!='PASS':
    raise RuntimeError(f'STATIC PREFLIGHT FAIL: {failed}')
if static['pretrained_checkpoint_sha256']!=PRETRAIN_SHA:
    raise RuntimeError('Checkpoint SHA changed between download and preflight')
if (DATA/'test').exists():
    raise RuntimeError('TEST TEREXPOSE — STOP')
print('STATIC PREFLIGHT PASS')
print('PARAMS native/candidate:',static['native_parameter_count'],static['candidate_parameter_count'])
print('COMMON INIT SHA:',static['common_initialized_detector_state_sha256'])
print('CANDIDATE INIT SHA:',static['candidate_initialized_detector_state_sha256'])
print('AF2 PARAMS:',static['af2_learned_parameter_count'])


In [ ]:
def log_last_epoch(run_dir):
    path=Path(run_dir)/'log.txt'
    if not path.is_file():
        return -1
    last=-1
    for line in path.read_text(errors='replace').splitlines():
        try:
            last=max(last,int(json.loads(line).get('epoch',-1)))
        except Exception:
            pass
    return last

def snapshot_state():
    if STATE_ZIP.exists():
        STATE_ZIP.unlink()
    archive=Path(shutil.make_archive(
        str(STATE_ZIP.with_suffix('')),'zip',root_dir=WORK,base_dir=OUT.name
    ))
    print('STATE SNAPSHOT:',archive,archive.stat().st_size,'bytes')
    return archive

def train_arm(label,cfg):
    run_dir=OUT/f'{label}_seed42'
    last_epoch=log_last_epoch(run_dir)
    if last_epoch>=219:
        print(label,'already complete:',last_epoch)
        return

    last_ckpt=run_dir/'last.pth'
    cmd=[sys.executable,'-u','train.py','-c',str(cfg),'--use-amp','--seed=42']
    if last_ckpt.is_file():
        cmd += ['-r',str(last_ckpt)]
        print(label,'RESUME from',last_ckpt,'logged epoch',last_epoch)
    else:
        cmd += ['-t',str(PRETRAIN)]
        print(label,'START from official D-FINE-N COCO')

    log_path=OUT/f'{label}_console.log'
    with log_path.open('a',encoding='utf-8') as stream:
        proc=subprocess.run(cmd,cwd=DFINE,stdout=stream,stderr=subprocess.STDOUT)
    if proc.returncode:
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-180:]))
        snapshot_state()
        raise RuntimeError(f'{label} training failed rc={proc.returncode}')
    print(label,'DONE, last logged epoch=',log_last_epoch(run_dir))
    snapshot_state()

train_arm('DFN0',NATIVE_CFG)
train_arm('DFN_AF2',AF2_CFG)


In [ ]:
def select_official_best(run_dir):
    run_dir=Path(run_dir)
    stg2=run_dir/'best_stg2.pth'
    stg1=run_dir/'best_stg1.pth'
    if stg2.is_file():
        return stg2
    if stg1.is_file():
        return stg1
    raise FileNotFoundError(f'No official D-FINE best checkpoint under {run_dir}')

def evaluate_arm(label,cfg):
    run_dir=OUT/f'{label}_seed42'
    best=select_official_best(run_dir)
    eval_path=run_dir/'eval.pth'
    cmd=[
        sys.executable,'-u','train.py','-c',str(cfg),
        '--test-only','-r',str(best),'--seed=42'
    ]
    log_path=OUT/f'{label}_eval_console.log'
    with log_path.open('w',encoding='utf-8') as stream:
        proc=subprocess.run(cmd,cwd=DFINE,stdout=stream,stderr=subprocess.STDOUT)
    if proc.returncode:
        print('\n'.join(log_path.read_text(errors='replace').splitlines()[-180:]))
        raise RuntimeError(f'{label} validation failed rc={proc.returncode}')
    if not eval_path.is_file():
        raise FileNotFoundError(eval_path)
    print(label,'BEST:',best.name,'EVAL:',eval_path)
    return eval_path,best

CONTROL_EVAL,CONTROL_BEST=evaluate_arm('DFN0',NATIVE_CFG)
AF2_EVAL,AF2_BEST=evaluate_arm('DFN_AF2',AF2_CFG)
snapshot_state()


In [ ]:
SUMMARY=OUT/'af2_dfine_seed42_summary.json'
VAL_ANN=Path(prep['dataset']['splits']['val']['path'])
cmd=[
    sys.executable,str(COFFEE/'scripts/analyze_dfine_af2_transfer.py'),
    '--control-eval',str(CONTROL_EVAL),
    '--candidate-eval',str(AF2_EVAL),
    '--val-annotations',str(VAL_ANN),
    '--output',str(SUMMARY),
]
r=subprocess.run(cmd,cwd=COFFEE,text=True,capture_output=True)
if r.returncode:
    print(r.stdout); print(r.stderr)
    raise RuntimeError('ANALYSIS FAILED')
summary=json.loads(SUMMARY.read_text())
snapshot_state()

from IPython.display import display
import pandas as pd

c=summary['control']; a=summary['candidate']; d=summary['deltas_af2_minus_native']
rows=[
    {'metric':'Macro class AP50-95','DFN0':c['macro_class_ap50_95'],'DFN_AF2':a['macro_class_ap50_95'],'delta':d['macro']},
    {'metric':'Bottom-3 class AP50-95','DFN0':c['bottom3_class_ap50_95'],'DFN_AF2':a['bottom3_class_ap50_95'],'delta':d['bottom3']},
    {'metric':'Worst-class AP50-95','DFN0':c['worst_class_ap50_95'],'DFN_AF2':a['worst_class_ap50_95'],'delta':d['worst']},
    {'metric':'COCO AP50-95 from precision','DFN0':c['global_coco_ap_from_precision'],'DFN_AF2':a['global_coco_ap_from_precision'],'delta':d['global_coco_ap']},
]
display(pd.DataFrame(rows).style.format({'DFN0':'{:.2%}','DFN_AF2':'{:.2%}','delta':d['global_coco_ap']}))
print('SCREEN:',json.dumps(summary['screen'],indent=2))
print('PRETRAIN SHA256:',PRETRAIN_SHA)
print('COMMON INITIAL DETECTOR SHA:',static['common_initialized_detector_state_sha256'])
print('CONTROL BEST:',CONTROL_BEST)
print('AF2 BEST:',AF2_BEST)
print('TEST:',summary['test_images_accessed'])
print('STATE ZIP:',STATE_ZIP)
print('Kirim tabel + SCREEN + PRETRAIN SHA. Jangan buka test.')
